# Tutorial 5: Comparing Results

**Analysis techniques for multiple calculations**

## 🎓 Learning Objectives

By the end of this tutorial, you will:
- ✅ Run a small, systematic *series* of calculations from Python
- ✅ Save results to disk and reload them for side-by-side comparison
- ✅ Decide which comparisons are physically meaningful (and which are not)
- ✅ Compute energy *differences* (correlation energy, ionization energy) and see why they converge faster than total energies
- ✅ Visualize trends and export a summary table (CSV)

## ⏱️ Estimated Time
**25-30 minutes** (the calculations themselves take well under a minute on a laptop CPU)

## 📚 Prerequisites
- Tutorials 1-4 are helpful (basis sets, open-shell/UHF, charged species), but not required
- PySCF installed (Linux, macOS, or WSL)

---

**Let's analyze and compare results!** 📊

---

## Introduction: Why Systematic Comparison Matters

### Research Workflow
1. Run a **planned series** of calculations (different molecules, methods, basis sets)
2. **Save** every result with its full specification (method, basis, charge, multiplicity)
3. **Compare** like with like and identify patterns
4. **Visualize** for understanding and communication
5. **Report** findings with all computational details

### The golden rule of comparison
A total electronic energy (in Hartree) is only meaningful **relative to another energy computed with the same method and the same basis set**.
- Comparing total energies of *different molecules* tells you nothing on its own (more electrons → much lower energy).
- Comparing total energies from *different basis sets or methods* tells you how the *calculation* changes, not something about the chemistry.
- **Energy differences** between related calculations at the *same* level of theory (reaction energies, ionization energies, spin-state gaps) are the chemically meaningful numbers.

### What we'll compare today
- **Part A:** Water (H₂O), RHF, five basis sets → basis-set convergence of the total energy and properties
- **Part B:** Ammonia (NH₃) and its cation (NH₃⁺) → the vertical ionization energy at each basis set
- **Part C:** Water at a fixed basis set with three methods (RHF, MP2, B3LYP)

All calculations run directly in this notebook's kernel with `quantui.run_in_session`.

---

## Step 1: Setup

Import QuantUI, check that PySCF is available, and create a **temporary results folder** for this tutorial.

The tutorial saves its results into that temporary folder (not your real `results/` folder), so it will not clutter the History tab of the QuantUI app.

### Run the cell below ⬇️

In [ ]:
import csv
import io
import shutil
import statistics
import tempfile
from pathlib import Path

from IPython.display import HTML, display

import quantui
from quantui import (
    Molecule,
    comparison_table_html,
    list_results,
    load_result,
    parse_xyz_input,
    plot_comparison,
    save_result,
    summary_from_saved_result,
)

try:
    import pyscf
    from quantui import run_in_session

    PYSCF_AVAILABLE = True
except ImportError:
    PYSCF_AVAILABLE = False
    print("PySCF not available — calculations will not run.")
    print("PySCF requires Linux, macOS, or WSL.")

# Unit conversions (CODATA 2018)
HARTREE_TO_EV = 27.211386245988
HARTREE_TO_KCAL = 627.509474

# Tutorial-local results folder (deleted again in the last step)
RESULTS_DIR = Path(tempfile.mkdtemp(prefix="quantui_tutorial05_"))

print(f"✓ QuantUI {quantui.__version__} loaded")
if PYSCF_AVAILABLE:
    print(f"✓ PySCF {pyscf.__version__} available")
print(f"✓ Tutorial results folder: {RESULTS_DIR}")

---

## Step 2: Build the Molecules

We use fixed, reasonable geometries (these are **not** optimized):
- **H₂O:** O–H = 0.957 Å, H–O–H = 104.5°
- **NH₃:** N–H = 1.012 Å, H–N–H = 106.7°

The NH₃⁺ cation uses **exactly the same geometry** as neutral NH₃. Removing an electron without letting the nuclei move gives the **vertical** ionization energy.

Note the charge and spin multiplicity: NH₃⁺ has 9 electrons, so it is a doublet (multiplicity 2) and needs an unrestricted (UHF) calculation — see Tutorials 3 and 4.

### Run the cell below ⬇️

In [ ]:
WATER_XYZ = '''
O   0.0000   0.0000   0.0000
H   0.7570   0.5859   0.0000
H  -0.7570   0.5859   0.0000
'''

AMMONIA_XYZ = '''
N   0.0000   0.0000   0.0000
H   0.9375   0.0000  -0.3810
H  -0.4688   0.8119  -0.3810
H  -0.4688  -0.8119  -0.3810
'''

atoms, coords = parse_xyz_input(WATER_XYZ)
water = Molecule(atoms, coords, charge=0, multiplicity=1)

atoms, coords = parse_xyz_input(AMMONIA_XYZ)
nh3 = Molecule(atoms, coords, charge=0, multiplicity=1)
nh3_cation = Molecule(atoms, coords, charge=1, multiplicity=2)

for name, mol in [("water", water), ("ammonia", nh3), ("ammonia cation", nh3_cation)]:
    print(f"{name:<15s} {mol.get_formula():<5s} charge = {mol.charge:+d}   multiplicity = {mol.multiplicity}")

---

## Step 3: A Helper That Runs *and* Saves a Calculation

When you run many calculations, write one small helper so that **every** calculation is run and recorded the same way. Ours:
1. runs the calculation with `run_in_session` (PySCF's log goes into a string instead of flooding the screen),
2. saves it with `save_result` — the same function the QuantUI app uses — together with the geometry and a readable label,
3. prints a one-line summary.

`warm_start=False` makes every SCF start from a fresh initial guess, so the iteration counts are comparable and nothing from earlier sessions influences the run.

### Run the cell below ⬇️

In [ ]:
def run_and_save(molecule, method, basis, label):
    # Run one calculation, save it to RESULTS_DIR, and print a summary line.
    log = io.StringIO()
    result = run_in_session(
        molecule,
        method=method,
        basis=basis,
        verbose=3,
        progress_stream=log,
        warm_start=False,
    )
    save_result(
        result,
        pyscf_log=log.getvalue(),
        results_dir=RESULTS_DIR,
        molecule=molecule,
        extras={"tutorial_label": label},
    )
    status = "converged" if result.converged else "NOT CONVERGED"
    print(f"{label:<24s} E = {result.energy_hartree:14.8f} Ha   ({status}, {result.n_iterations} iterations)")
    return result


BASIS_SETS = ["STO-3G", "6-31G", "6-31G*", "cc-pVDZ", "cc-pVTZ"]
print("Basis sets for Parts A and B:", ", ".join(BASIS_SETS))

---

## Step 4: Run the Calculation Series

### Part A — Water, RHF, five basis sets

### Run the cell below ⬇️

In [ ]:
water_rhf = {}
if PYSCF_AVAILABLE:
    for basis in BASIS_SETS:
        water_rhf[basis] = run_and_save(water, "RHF", basis, f"H2O RHF/{basis}")
else:
    print("⚠️  PySCF is not available — skipping.")

### Part B — NH₃ (RHF) and NH₃⁺ (UHF) at the same five basis sets

The neutral is closed-shell (RHF); the cation is open-shell (UHF). Both use the **same basis set** in each pair, so their energy difference is meaningful.

You may see a few `WARN: HOMO ... >= LUMO ...` lines from PySCF while the cation runs. They come from the *early* SCF iterations, when two degenerate orbitals have exactly the same energy but only one of them can be occupied. They are harmless as long as the calculation converges and ⟨S²⟩ is close to 0.75 (we check both below).

### Run the cell below ⬇️

In [ ]:
nh3_neutral = {}
nh3_plus = {}
if PYSCF_AVAILABLE:
    for basis in BASIS_SETS:
        nh3_neutral[basis] = run_and_save(nh3, "RHF", basis, f"NH3 RHF/{basis}")
        nh3_plus[basis] = run_and_save(nh3_cation, "UHF", basis, f"NH3+ UHF/{basis}")
else:
    print("⚠️  PySCF is not available — skipping.")

### Part C — Water at 6-31G* with three methods

We already have RHF/6-31G* from Part A, so we only add MP2 and B3LYP at the same basis set and geometry.

### Run the cell below ⬇️

In [ ]:
METHOD_BASIS = "6-31G*"
water_methods = {}
if PYSCF_AVAILABLE:
    water_methods["RHF"] = water_rhf[METHOD_BASIS]  # reuse Part A
    for method in ["MP2", "B3LYP"]:
        water_methods[method] = run_and_save(water, method, METHOD_BASIS, f"H2O {method}/{METHOD_BASIS}")
else:
    print("⚠️  PySCF is not available — skipping.")

### ✅ Checkpoint 1

Run the cell below to check your series.

- [ ] 17 calculations were saved (5 water + 10 ammonia + 2 extra water methods)
- [ ] Every calculation converged

**Always check convergence before comparing anything** — an unconverged energy is not a result.

In [ ]:
saved_dirs = list_results(RESULTS_DIR)
all_results = list(water_rhf.values()) + list(nh3_neutral.values()) + list(nh3_plus.values())
all_results += [water_methods[m] for m in ("MP2", "B3LYP") if m in water_methods]

print(f"Result folders saved: {len(saved_dirs)}")
print(f"All converged:        {all(r.converged for r in all_results) if all_results else 'n/a'}")
if saved_dirs:
    example = sorted(saved_dirs)[0]
    print("\nExample result folder:", example.name)
    for f in sorted(example.iterdir()):
        print("   ", f.name)

---

## Step 5: Reload Saved Results and Build a Comparison Table

Each saved calculation is a folder containing `result.json` (the numbers) and `pyscf.log` (the full PySCF output). We now **reload everything from disk** — exactly what the QuantUI app does:

- the app's **History** tab lists the saved results in your results folder,
- the app's **Compare** tab lets you tick several of them and shows this same table and chart (it is built on the same `comparison_table_html` / `plot_comparison` helpers used below).

### Run the cell below ⬇️

In [ ]:
# list_results returns newest first; reverse to get the order we ran them in
saved = [load_result(d) for d in reversed(list_results(RESULTS_DIR))]
summaries = [summary_from_saved_result(d, label=d.get("tutorial_label", "")) for d in saved]

print(f"Loaded {len(summaries)} saved results")
display(HTML(comparison_table_html(summaries)))

### 🤔 Which rows can be compared?

This table puts everything side by side, but most pairs of rows should **not** be compared directly:
- **H₂O vs NH₃ energies:** different molecules with different numbers of electrons — not comparable.
- **H₂O RHF/STO-3G vs H₂O RHF/cc-pVTZ:** same molecule and method, different basis — the difference measures the *basis-set error*, not chemistry.
- **NH₃ vs NH₃⁺ with the same basis set:** a meaningful difference — the ionization energy.
- **H₂O MP2 vs H₂O RHF with the same basis:** a meaningful difference — MP2 adds a correlation correction on top of the same RHF reference.

The next three steps make each kind of comparison properly.

---

## Step 6: Part A — Basis-Set Convergence (Water, RHF)

Same molecule, same geometry, same method — only the basis set changes. We report each energy **relative to the largest basis set (cc-pVTZ)**, together with two properties:
- the **dipole moment**, a physical observable, and
- the **Mulliken charge on oxygen**, which is *not* an observable (it depends on how electrons are partitioned between basis functions).

### Run the cell below ⬇️

In [ ]:
if water_rhf:
    e_ref = water_rhf["cc-pVTZ"].energy_hartree
    print("H2O, RHF — basis-set series (relative to cc-pVTZ)")
    print("-" * 86)
    print(f"{'Basis':<9s} {'E (Ha)':>14s} {'E - E(TZ) (Ha)':>15s} {'(kcal/mol)':>11s} "
          f"{'Gap (eV)':>9s} {'Dipole (D)':>11s} {'q(O) Mulliken':>14s}")
    print("-" * 86)
    for basis, r in water_rhf.items():
        de = r.energy_hartree - e_ref
        print(f"{basis:<9s} {r.energy_hartree:14.6f} {de:15.6f} {de * HARTREE_TO_KCAL:11.1f} "
              f"{r.homo_lumo_gap_ev:9.2f} {r.dipole_moment_debye:11.2f} {r.mulliken_charges[0]:14.2f}")
    print("-" * 86)

### What to notice

With the geometry above you should see:
- **The energy drops as the basis improves:** from −74.962935 Ha (STO-3G) to −76.057165 Ha (cc-pVTZ). STO-3G is about 1.09 Ha (≈ 687 kcal/mol) above cc-pVTZ; even cc-pVDZ is still about 19 kcal/mol above it.
- **Why it drops:** the variational principle. Going from 6-31G to 6-31G* only *adds* d functions, so the 6-31G* energy cannot be higher. (The guarantee is strict only when one basis set contains the other, but larger basis sets of good quality reliably give lower HF energies.)
- **The dipole moment settles down:** about 2.63 D (6-31G), 2.22 D (6-31G*), 2.06 D (cc-pVDZ), 2.02 D (cc-pVTZ). Note the large jump when polarization (d) functions are added to 6-31G.
- **The Mulliken charge on O jumps around** (from about −0.31 to −0.90 e) with **no** trend toward a limit — a warning that Mulliken charges should only be compared within one basis set.

Now let's visualize it. The first figure uses QuantUI's built-in `plot_comparison` (the same chart as the app's Compare tab); the second plots energy **differences**.

### Run the cell below ⬇️

In [ ]:
import matplotlib.pyplot as plt

water_summaries = [s for s in summaries if s.label.startswith("H2O RHF/")]
if water_summaries:
    fig = plot_comparison(water_summaries, title="H2O RHF — basis-set series")
    plt.show()

    # Energy relative to cc-pVTZ, on a log scale (STO-3G is far above the rest)
    basis_labels = [b for b in BASIS_SETS if b != "cc-pVTZ"]
    de_kcal = [(water_rhf[b].energy_hartree - e_ref) * HARTREE_TO_KCAL for b in basis_labels]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(basis_labels, de_kcal, "o-", color="#2171b5")
    for x, y in zip(basis_labels, de_kcal):
        ax.annotate(f"{y:.1f}", (x, y), textcoords="offset points", xytext=(0, 8), ha="center")
    ax.set_yscale("log")
    ax.set_ylabel("E − E(cc-pVTZ)  (kcal/mol, log scale)")
    ax.set_xlabel("Basis set")
    ax.set_title("H2O RHF: energy above the cc-pVTZ result")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()

**Look at the left panel of the first figure.** The total-energy bars look almost identical, because differences of a fraction of a Hartree are tiny compared with 75-76 Ha. Plotting **differences** (second figure) shows the convergence clearly. Choose the quantity you plot as carefully as the calculations you run.

---

## Step 7: Part B — Ionization Energy of NH₃ (an Energy *Difference*)

The vertical ionization energy (IE) from two SCF calculations (the "ΔSCF" approach):

$$\text{IE} = E(\text{NH}_3^+) - E(\text{NH}_3)$$

Both energies must come from the **same basis set** (and here the same geometry). We compare how much the IE changes across basis sets with how much the *total* energy of NH₃ changes.

### Run the cell below ⬇️

In [ ]:
ie_ev = {}
if nh3_neutral:
    print("NH3 vertical ionization energy, ΔSCF (RHF neutral, UHF cation)")
    print("-" * 74)
    print(f"{'Basis':<9s} {'E(NH3) (Ha)':>14s} {'E(NH3+) (Ha)':>14s} {'IE (eV)':>9s} {'<S^2> cation':>14s}")
    print("-" * 74)
    for basis in BASIS_SETS:
        e0 = nh3_neutral[basis].energy_hartree
        e1 = nh3_plus[basis].energy_hartree
        ie_ev[basis] = (e1 - e0) * HARTREE_TO_EV
        s2 = nh3_plus[basis].spin_square
        s2_str = f"{s2:.4f}" if s2 is not None else "n/a"
        print(f"{basis:<9s} {e0:14.6f} {e1:14.6f} {ie_ev[basis]:9.2f} {s2_str:>14s}")
    print("-" * 74)
    print("Exact <S^2> for a doublet is S(S+1) = 0.75")

### Statistics: how sensitive is each quantity to the basis set?

STO-3G is a *minimal* basis and is clearly not converged, so we look at the spread over the four larger basis sets (6-31G through cc-pVTZ). A simple **range** (max − min) and **standard deviation** are enough here.

### Run the cell below ⬇️

In [ ]:
if ie_ev:
    larger = BASIS_SETS[1:]  # 6-31G ... cc-pVTZ
    e_nh3_ev = [nh3_neutral[b].energy_hartree * HARTREE_TO_EV for b in larger]
    ie_vals = [ie_ev[b] for b in larger]

    print(f"Over {', '.join(larger)}:")
    print(f"  Total energy of NH3: range = {max(e_nh3_ev) - min(e_nh3_ev):.2f} eV, "
          f"std dev = {statistics.stdev(e_nh3_ev):.2f} eV")
    print(f"  Ionization energy:   range = {max(ie_vals) - min(ie_vals):.2f} eV, "
          f"std dev = {statistics.stdev(ie_vals):.2f} eV, mean = {statistics.mean(ie_vals):.2f} eV")
    print(f"\n  STO-3G IE: {ie_ev['STO-3G']:.2f} eV  (minimal basis — far from the others)")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    e_ref_nh3 = nh3_neutral["cc-pVTZ"].energy_hartree
    axes[0].plot(larger, [(nh3_neutral[b].energy_hartree - e_ref_nh3) * HARTREE_TO_EV for b in larger],
                 "o-", color="#2171b5")
    axes[0].set_ylabel("E(NH3) − E(NH3, cc-pVTZ)  (eV)")
    axes[0].set_title("Total energy of NH3")
    axes[1].plot(larger, [ie_ev[b] - ie_ev["cc-pVTZ"] for b in larger], "o-", color="#e6550d")
    axes[1].set_ylabel("IE − IE(cc-pVTZ)  (eV)")
    axes[1].set_title("Ionization energy")
    lo = min(ax.get_ylim()[0] for ax in axes)
    hi = max(ax.get_ylim()[1] for ax in axes)
    for ax in axes:
        ax.set_ylim(lo, hi)  # same scale on both panels for a fair visual comparison
        ax.axhline(0, color="gray", lw=0.8)
        ax.grid(alpha=0.3)
    fig.suptitle("Same y-axis scale: the energy difference is far less basis-sensitive")
    fig.tight_layout()
    plt.show()

### What to notice

- From 6-31G to cc-pVTZ, the **total energy of NH₃ changes by about 1.55 eV**, but the **ionization energy changes by only about 0.18 eV** (9.15 to 9.33 eV). Much of the basis-set error is similar for NH₃ and NH₃⁺, so it largely **cancels** in the difference. This error cancellation is why chemists compute energy differences, not total energies.
- The IE does **not** change monotonically (cc-pVDZ gives a slightly *lower* IE than 6-31G*). Energy differences are not protected by the variational principle, so they can converge from either side.
- STO-3G gives about 7.66 eV — a minimal basis is too inflexible even for energy differences.
- ⟨S²⟩ for the cation is about 0.757, very close to the exact 0.75: spin contamination is negligible, so the UHF cation energy is trustworthy.
- Converging the basis set does **not** fix the *method*. Hartree-Fock neglects electron correlation, and the neutral (10 electrons) has more correlation energy than the cation (9 electrons), so ΔSCF-HF tends to **underestimate** ionization energies. Compare with experiment in the discussion questions.

---

## Step 8: Part C — Comparing Methods at a Fixed Basis Set

Same molecule, geometry and basis set (6-31G*); three methods. Which differences mean something?

### Run the cell below ⬇️

In [ ]:
if water_methods:
    print(f"H2O, {METHOD_BASIS} — method comparison")
    print("-" * 52)
    print(f"{'Method':<8s} {'E (Ha)':>14s} {'Gap (eV)':>10s} {'Dipole (D)':>12s}")
    print("-" * 52)
    for method, r in water_methods.items():
        print(f"{method:<8s} {r.energy_hartree:14.6f} {r.homo_lumo_gap_ev:10.2f} {r.dipole_moment_debye:12.2f}")
    print("-" * 52)

    mp2 = water_methods["MP2"]
    e_corr = mp2.energy_hartree - water_methods["RHF"].energy_hartree
    print(f"\nMP2 correlation energy = E(MP2) - E(RHF) = {e_corr:.6f} Ha = {e_corr * HARTREE_TO_KCAL:.1f} kcal/mol")
    print(f"(QuantUI also reports it directly: mp2_correlation_hartree = {mp2.mp2_correlation_hartree:.6f} Ha)")

### What to notice

- **RHF vs MP2 — meaningful.** MP2 starts from the RHF wavefunction and adds a correlation correction, so E(MP2) − E(RHF) is the MP2 correlation energy: about −0.186 Ha (≈ −117 kcal/mol) for water at 6-31G*. Notice that this is larger than many reaction energies — correlation matters.
- **The MP2 gap and dipole are identical to RHF** (19.38 eV, 2.22 D). That is not a coincidence: QuantUI reports these from the RHF reference orbitals and density, because MP2 changes the energy, not the orbitals. Always know *where* a reported number comes from.
- **B3LYP's total energy (−76.406787 Ha) is lower than MP2's, but that does not make it "better".** DFT total energies are not on the same scale as HF/MP2 total energies; only differences computed with B3LYP throughout (e.g. a B3LYP ionization energy) are meaningful.
- **The HOMO-LUMO gap is strongly method-dependent:** about 19.38 eV with RHF vs 9.78 eV with B3LYP. Orbital gaps should only be compared within one method (and basis set).

### ✅ Checkpoint 2

- [ ] You can explain why E(MP2) − E(RHF) is meaningful but E(B3LYP) − E(MP2) is not
- [ ] You can explain why the ionization energy converges faster with basis set than the total energy
- [ ] You know which of your properties are observables (energy differences, dipole moment) and which are not (Mulliken charges)

---

## Step 9: Export Data for External Analysis

A CSV file can be opened in Excel/LibreOffice or read by any analysis tool. Python's built-in `csv` module is all you need. We write it into the tutorial's temporary folder; to keep a copy, change `csv_file` to a path of your choice.

### Run the cell below ⬇️

In [ ]:
csv_file = RESULTS_DIR / "tutorial05_summary.csv"
columns = ["label", "formula", "charge", "multiplicity", "method", "basis",
           "energy_hartree", "homo_lumo_gap_ev", "dipole_moment_debye", "converged", "n_iterations"]

with open(csv_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(columns)
    for d in saved:
        geom = d.get("geometry", {})
        writer.writerow([
            d.get("tutorial_label", ""), d["formula"], geom.get("charge"), geom.get("multiplicity"),
            d["method"], d["basis"], d["energy_hartree"], d["homo_lumo_gap_ev"],
            d["dipole_moment_debye"], d["converged"], d["n_iterations"],
        ])

print(f"✓ Wrote {len(saved)} rows to {csv_file}\n")
print("First lines of the file:")
with open(csv_file) as f:
    for line in f.readlines()[:4]:
        print("  ", line.rstrip())

Note that the CSV records **charge and multiplicity** as well as method and basis. Without them, the NH₃ and NH₃⁺ rows would share the formula `H3N` and could be confused. (The dipole moment of a *charged* species depends on the choice of origin, so do not compare the NH₃⁺ dipole with anything.)

---

## Step 10: A Publication-Style Summary

A results table should say exactly how every number was obtained.

### Run the cell below ⬇️

In [ ]:
if ie_ev:
    print("Table 1. Vertical ionization energy of NH3 (ΔSCF, fixed geometry).")
    print("-" * 46)
    print(f"{'Basis set':<12s} {'IE (eV)':>10s} {'IE - IE(cc-pVTZ) (eV)':>22s}")
    print("-" * 46)
    for basis in BASIS_SETS:
        print(f"{basis:<12s} {ie_ev[basis]:10.2f} {ie_ev[basis] - ie_ev['cc-pVTZ']:22.2f}")
    print("-" * 46)

    print("\nComputational details (methods statement):")
    print(
        f"Single-point calculations were performed with PySCF {pyscf.__version__} through "
        f"QuantUI {quantui.__version__}. NH3 (N-H 1.012 Å, H-N-H 106.7°, not optimized) was "
        "treated with restricted Hartree-Fock and NH3+ (doublet) with unrestricted Hartree-Fock "
        "at the same geometry, using the STO-3G, 6-31G, 6-31G*, cc-pVDZ and cc-pVTZ basis sets. "
        "Vertical ionization energies were obtained as E(NH3+) - E(NH3). "
        + ("All SCF calculations converged. " if all(r.converged for r in all_results)
           else "WARNING: not all SCF calculations converged. ")
        + "1 Hartree = 27.2114 eV."
    )

---

## Step 11: Clean Up

Delete the tutorial's temporary results folder. Set `KEEP_FILES = True` first if you want to look inside `result.json` or `pyscf.log` (the folder path is printed in Step 1).

### Run the cell below ⬇️

In [ ]:
KEEP_FILES = False

if KEEP_FILES:
    print(f"Kept tutorial results in {RESULTS_DIR}")
else:
    shutil.rmtree(RESULTS_DIR, ignore_errors=True)
    print(f"✓ Removed {RESULTS_DIR}")

---

## 🎓 Learning Review

### What You Accomplished

1. ✅ Ran a planned series of 17 calculations from Python
2. ✅ Saved them with `save_result` and reloaded them with `list_results` / `load_result`
3. ✅ Built comparison tables and charts with QuantUI's comparison helpers
4. ✅ Measured basis-set convergence of energies and properties
5. ✅ Computed meaningful energy differences (ionization energy, correlation energy)
6. ✅ Exported a CSV file and wrote a methods statement

### Key Skills Learned

**Comparison rules:**
- Compare total energies only at the same method *and* basis set — and then only as differences between related systems
- Energy differences benefit from error cancellation and converge faster than total energies
- Know which properties are observables (energy differences, dipole moments) and which depend on arbitrary choices (Mulliken charges)

**Data management:**
- One helper function so every calculation is run and recorded identically
- Record charge and multiplicity, not just formula, method and basis
- Check convergence before comparing

**Visualization:**
- Plot differences, not raw totals, when the totals are huge
- Use a common axis scale when comparing sensitivities
- Use a log scale when values span orders of magnitude

### Best Practices

**Comparison:**
- Same geometry, method and basis set for every member of a comparison
- Converge the basis set for the *quantity you report*, not for the total energy
- A converged basis set does not fix the limitations of the method

**Reporting:**
- State method, basis set, geometry source, charge/multiplicity and software versions
- Report energy differences in chemically meaningful units (eV, kcal/mol, kJ/mol)
- Cite the software (PySCF, QuantUI)

---

## 🚀 Advanced Analysis Ideas

1. **More basis sets:** add `aug-cc-pVDZ` (diffuse functions) to Part B. Does the IE change much? Why might diffuse functions matter for the *cation vs neutral* difference?
2. **Correlated ionization energy:** repeat Part B with MP2 (MP2 for both NH₃ and NH₃⁺). Does correlation increase or decrease the IE?
3. **DFT ionization energy:** repeat Part B with B3LYP for both species and compare with the HF and MP2 values.
4. **Basis-set extrapolation:** add cc-pVQZ to Part A and fit the cc-pVDZ/TZ/QZ water energies to estimate the complete-basis-set (CBS) limit.
5. **Adiabatic vs vertical:** optimize the NH₃⁺ geometry (it becomes planar) in the main app and compare the adiabatic IE with the vertical one.

---

## 💬 Discussion Questions

1. **Error cancellation:** From 6-31G to cc-pVTZ, the NH₃ total energy changed by about 1.55 eV but the ionization energy by only about 0.18 eV. Explain why. Would you expect the same cancellation for a reaction that breaks a bond?

2. **Experiment:** Look up the experimental vertical ionization energy of NH₃ and the experimental dipole moment of H₂O (for example in the NIST Computational Chemistry Comparison and Benchmark Database, CCCBDB). How large are the HF errors? Is the remaining error due to the basis set or the method?

3. **Variational principle:** Why must the 6-31G* energy be at or below the 6-31G energy? Why is there no such guarantee for the ionization energy?

4. **Comparing methods:** A classmate says "B3LYP gave the lowest energy, so it is the most accurate method." What is wrong with this argument?

5. **Mulliken charges:** The Mulliken charge on oxygen varied from about −0.31 to −0.90 e across the basis sets, while the dipole moment converged smoothly. What does that tell you about using Mulliken charges in a report?

6. **Outliers:** If one calculation in a series gives a very different result than expected, what would you check first? (Hint: convergence, charge/multiplicity, ⟨S²⟩, geometry.)

---

## 📚 Further Reading

### Data Analysis in Computational Chemistry
- Cramer, *Essentials of Computational Chemistry* — basis sets, correlation and error analysis
- Jensen, *Introduction to Computational Chemistry* — basis-set convergence and extrapolation

### Benchmark Data
- NIST Computational Chemistry Comparison and Benchmark Database (CCCBDB)
- Benchmark sets: G2, G3, G4 test sets

### Statistical Methods and Benchmarking
- Peverati & Truhlar, "Quest for a universal density functional" (*Phil. Trans. R. Soc. A*, 2014)
- Goerigk & Grimme, benchmark studies of density functionals (GMTKN databases)

### Visualization
- Matplotlib documentation

---

## ✅ Tutorial Series Complete!

**Congratulations!** 🎉 You've completed all five QuantUI tutorials!

### What You've Mastered

1. **Tutorial 1:** First calculations and basics
2. **Tutorial 2:** Basis set convergence
3. **Tutorial 3:** Multiplicity and radicals
4. **Tutorial 4:** Charged species
5. **Tutorial 5:** Results analysis and comparison

### Next Steps

**Open the main app:** [molecule_computations.ipynb](../molecule_computations.ipynb)
- Run calculations interactively — they are saved automatically
- **History** tab: browse and reload every saved calculation
- **Compare** tab: select several saved calculations to see the same comparison table and chart you built here

**Read the documentation:** [docs/](../../docs/)

---

**Questions or feedback?** Contact your instructor or open a GitHub issue.

**Happy calculating!** 🚀

---

*End of Tutorial Series*